In [1]:
#Data Engineering
import json
import pandas as pd

DATA_DIR = "/kaggle/input/competitions/fds-pokemon-battles-prediction-2026/"
train_path = DATA_DIR + "train.jsonl"
test_path = DATA_DIR + "test.jsonl"

import pandas as pd
chart_df = pd.read_csv(DATA_DIR + "type_chart_gen2.csv")
chart = {(a, d): m for a, d, m in chart_df.itertuples(index=False)}

def type_mult(atk_type, def_types):
    mult = 1.0
    for t in def_types:
        mult *= chart.get((atk_type, t), 1.0)
    return mult

def get_imp_info(battle):
    p1_endlife = {}
    p2_endlife = {}
    psn_p1 = par_p1 = slp_p1 = 0
    psn_p2 = par_p2 = slp_p2 = 0
    total_boost_p1 = total_boost_p2 = 0
    switch_p1 = switch_p2 = 0
    previous_poke1 = previous_poke2 = None
    pre_poke_life1 = pre_poke_life2 = 1
    forced_p1 = forced_p2 = 0
    stat_move1 = hit_move1 = power_sum1 = recov_move1 = 0
    stat_move2 = hit_move2 = power_sum2 = recov_move2 = 0
    moves_p1 = {}
    moves_p2 = {}

    for turn in battle["battle_timeline"]:
        s1 = turn["p1_pokemon_state"]
        s2 = turn["p2_pokemon_state"]
        p1_endlife[s1["name"]] = s1["hp_pct"]
        p2_endlife[s2["name"]] = s2["hp_pct"]
        total_boost_p1 += sum(s1["boosts"].values())
        total_boost_p2 += sum(s2["boosts"].values())
        turn_poke1 = s1["name"]
        turn_poke2 = s2["name"]
        m1 = turn["p1_move_details"]
        m2 = turn["p2_move_details"]

        if m1 is not None:
            moves_p1[m1["name"]] = moves_p1.get(m1["name"], 0) + 1
            if m1["category"] == "status":
                stat_move1 += 1
            else:
                hit_move1 += 1
                power_sum1 += m1["base_power"]

            if m1["name"] in ("rest", "recover", "softboiled", "milkdrink", "moonlight", "morningsun", "synthesis"): recov_move1 += 1

        if m2 is not None:
            moves_p2[m2["name"]] = moves_p2.get(m2["name"], 0) + 1
            if m2["category"] == "status":
                stat_move2 += 1
            else:
                hit_move2 += 1
                power_sum2 += m2["base_power"]

            if m2["name"] in ("rest", "recover", "softboiled", "milkdrink", "moonlight", "morningsun", "synthesis"): recov_move2 += 1

        if previous_poke1 is not None and previous_poke1 != turn_poke1:
            switch_p1 += 1
            if pre_poke_life1 == 0:
                forced_p1 += 1

        if previous_poke2 is not None and previous_poke2 != turn_poke2:
            switch_p2 += 1
            if pre_poke_life2 == 0:
                forced_p2 += 1

        if s1["status"] == "slp":
            slp_p1 += 1
        elif s1["status"] == "par":
            par_p1 += 1
        elif s1["status"] in ("psn", "tox"):
            psn_p1 += 1

        if s2["status"] == "slp":
            slp_p2 += 1
        elif s2["status"] == "par":
            par_p2 += 1
        elif s2["status"] in ("psn", "tox"):
            psn_p2 += 1

        previous_poke1 = s1["name"]
        previous_poke2 = s2["name"]
        pre_poke_life1 = s1["hp_pct"]
        pre_poke_life2 = s2["hp_pct"]

    p1_dead = sum(1 for hp in p1_endlife.values() if hp == 0)
    p2_dead = sum(1 for hp in p2_endlife.values() if hp == 0)
    last = battle["battle_timeline"][-1]
    l1 = last["p1_pokemon_state"]
    l2 = last["p2_pokemon_state"]
    last_slp_p1 = 1 if l1["status"] == "slp" else 0
    last_slp_p2 = 1 if l2["status"] == "slp" else 0
    last_par_p1 = 1 if l1["status"] == "par" else 0
    last_par_p2 = 1 if l2["status"] == "par" else 0
    boost_mean_p1 = total_boost_p1 / len(battle["battle_timeline"])
    boost_mean_p2 = total_boost_p2 / len(battle["battle_timeline"])

    return {"hp_p1": sum(p1_endlife.values()) + (6 - len(p1_endlife)),
            "hp_p2": sum(p2_endlife.values()) + (6 - len(p2_endlife)),
            "dead_p1": p1_dead, "dead_p2": p2_dead,
            "slp_p1": slp_p1, "par_p1": par_p1, "psn_p1": psn_p1,
            "slp_p2": slp_p2, "par_p2": par_p2, "psn_p2": psn_p2,
            "last_hp_p1": l1["hp_pct"], "last_hp_p2": l2["hp_pct"],
            "last_boost_sum_p1": sum(l1["boosts"].values()), "last_boost_sum_p2": sum(l2["boosts"].values()),
            "last_slp_p1": last_slp_p1, "last_slp_p2": last_slp_p2,
            "last_par_p1": last_par_p1, "last_par_p2": last_par_p2,
            "boost_mean_p1" : boost_mean_p1, "boost_mean_p2": boost_mean_p2,
            "switch_p1": switch_p1, "switch_p2": switch_p2,
            "forced_p1": forced_p1, "forced_p2": forced_p2,
            "stat_move_p1": stat_move1, "hit_move_p1": hit_move1,
            "power_sum_p1": power_sum1, "recov_move_p1": recov_move1,
            "stat_move_p2": stat_move2, "hit_move_p2": hit_move2,
            "power_sum_p2": power_sum2, "recov_move_p2": recov_move2,
            "moves_p1": moves_p1, "moves_p2": moves_p2}
# def lead_features(battle):
    
def build_table(path):
    # Same feature function for train and test; label is added only if present
    rows = []
    with open(path) as f:
        for line in f:
            battle = json.loads(line)
            features = get_imp_info(battle)
            features["battle_id"] = battle["battle_id"]
            if "player_won" in battle:
                features["won"] = battle["player_won"]
            rows.append(features)
    return pd.DataFrame(rows)

train_df = build_table(train_path)
test_df = build_table(test_path)

# Move-usage columns: vocabulary (top 40 moves) is built from TRAIN only, then reused for test
from collections import Counter
move_counter = Counter()
for d in train_df["moves_p1"]:
    move_counter.update(d)
for d in train_df["moves_p2"]:
    move_counter.update(d)
top_moves = [m for m, _ in move_counter.most_common(40)]

def add_move_columns(df, top_moves):
    df = df.copy()
    for side in ("p1", "p2"):
        for m in top_moves:
            df[f"mv_{side}_{m}"] = df[f"moves_{side}"].apply(lambda d: d.get(m, 0))
    return df.drop(columns=["moves_p1", "moves_p2"])

train_df = add_move_columns(train_df, top_moves)
test_df = add_move_columns(test_df, top_moves)

# Feature columns: everything except id and label
feature_cols = [c for c in train_df.columns if c not in ("battle_id", "won")]

X = train_df[feature_cols]
y = train_df["won"]
X_test = test_df[feature_cols]

print(train_df.shape, test_df.shape)
print(X.columns.tolist() == X_test.columns.tolist())
print(X_test.isna().sum().sum())

(10000, 114) (5000, 113)
True
0


In [2]:
#Cell B: new features (type chart damage, matchups, categoricals). No labels are used anywhere in this cell.
import json, collections, os
import numpy as np
import pandas as pd

TYPE_CHART_PATH = DATA_DIR + "type_chart_gen2.csv"   # the file given by the competition
STAGE = {-6:.25, -5:.28, -4:.33, -3:.4, -2:.5, -1:.66, 0:1, 1:1.5, 2:2, 3:2.5, 4:3, 5:3.5, 6:4}

def load_chart(path):
    """Reads the competition type chart. Accepts long format (attacker, defender, multiplier) or a wide matrix."""
    df = pd.read_csv(path)
    df.columns = [str(c).strip().lower() for c in df.columns]
    chart = {}
    if df.shape[1] == 3 and df.iloc[:, 2].dtype.kind in "fi":
        for a, d, m in df.itertuples(index=False):
            chart[(str(a).strip().lower(), str(d).strip().lower())] = float(m)
    else:
        a_col = df.columns[0]
        for _, r in df.iterrows():
            for d in df.columns[1:]:
                chart[(str(r[a_col]).strip().lower(), d)] = float(r[d])
    assert chart.get(("water", "fire"), 1.0) == 2.0 and chart.get(("normal", "ghost"), 1.0) == 0.0, "type chart looks wrong"
    return chart

def tmult(chart, atk, def_types):
    m = 1.0
    for d in def_types:
        m *= chart.get((atk, d), 1.0)
    return m

def build_dex(path):
    """species -> base stats and types, taken from team details of the TRAIN file (no labels)."""
    dex = {}
    with open(path) as f:
        for line in f:
            b = json.loads(line)
            for p in b["p1_team_details"] + [b["p2_lead_details"]]:
                if p["name"] not in dex:
                    s = {k: float(p["base_" + k]) for k in ("hp", "atk", "def", "spa", "spd", "spe")}
                    s["types"] = [t for t in p["types"] if t != "notype"]
                    s["bst"] = sum(s[k] for k in ("hp", "atk", "def", "spa", "spd", "spe"))
                    dex[p["name"]] = s
    mean = {k: float(np.mean([v[k] for v in dex.values()])) for k in ("hp", "atk", "def", "spa", "spd", "spe", "bst")}
    mean["types"] = []
    return dex, mean

def stat(base):                      # level 100, max DV and stat exp
    return 2 * base + 98

def dmg_frac(mv, atk, dfn, ab, db, a_status, chart):
    """Estimated share of the defender's max HP removed by one use of a damaging move."""
    if not mv["base_power"]:
        return 0.0, 1.0
    mult = tmult(chart, mv["type"], dfn["types"])
    stab = 1.5 if mv["type"] in atk["types"] else 1.0
    if mv["category"] == "physical":
        A = stat(atk["atk"]) * STAGE[max(-6, min(6, ab.get("atk", 0)))]
        Dd = stat(dfn["def"]) * STAGE[max(-6, min(6, db.get("def", 0)))]
        if a_status == "brn":
            A *= 0.5
    else:
        A = stat(atk["spa"]) * STAGE[max(-6, min(6, ab.get("spa", 0)))]
        Dd = stat(dfn["spd"]) * STAGE[max(-6, min(6, db.get("spd", 0)))]
    raw = (42 * mv["base_power"] * A / max(Dd, 1) / 50 + 2) * stab * mult * 0.925 * (mv["accuracy"] or 1.0)
    return raw / (2 * dfn["hp"] + 203), mult

def spe_eff(info, boosts, status):
    return stat(info["spe"]) * STAGE[max(-6, min(6, boosts.get("spe", 0)))] * (0.25 if status == "par" else 1.0)

def new_features(b, dex, mean, chart):
    d, cats = {}, {}
    D = lambda n: dex.get(n, mean)
    team = b["p1_team_details"]; lead2 = b["p2_lead_details"]; tl = b["battle_timeline"]
    team_names = [p["name"] for p in team]

    # team preview: p1 team against the p2 lead
    l2 = D(lead2["name"])
    adv = [max([tmult(chart, t, l2["types"]) for t in D(n)["types"]] or [1.0]) for n in team_names]
    dis = [max([tmult(chart, t, D(n)["types"]) for t in l2["types"]] or [1.0]) for n in team_names]
    d["pre_adv_max"] = max(adv); d["pre_adv_mean"] = float(np.mean(adv))
    d["pre_dis_max"] = max(dis); d["pre_dis_mean"] = float(np.mean(dis))
    d["pre_faster_cnt"] = sum(D(n)["spe"] > l2["spe"] for n in team_names)
    d["pre_lead_adv"] = adv[0] - dis[0]

    seen_moves = {s: collections.defaultdict(dict) for s in ("p1", "p2")}
    last_hp = {s: {} for s in ("p1", "p2")}; last_status = {s: {} for s in ("p1", "p2")}
    dm = {s: [] for s in ("p1", "p2")}
    se = {s: 0 for s in ("p1", "p2")}; res = {s: 0 for s in ("p1", "p2")}
    imm = {s: 0 for s in ("p1", "p2")}; stab_n = {s: 0 for s in ("p1", "p2")}
    diffs = []
    for t in tl:
        st = {s: t[s + "_pokemon_state"] for s in ("p1", "p2")}
        for s in ("p1", "p2"):
            last_hp[s][st[s]["name"]] = st[s]["hp_pct"]; last_status[s][st[s]["name"]] = st[s]["status"]
        for s, o in (("p1", "p2"), ("p2", "p1")):
            mv = t[s + "_move_details"]
            if mv is not None:
                seen_moves[s][st[s]["name"]][mv["name"]] = mv
            a = D(st[s]["name"]); dfn = D(st[o]["name"])
            if mv is not None and mv["base_power"]:
                f, mult = dmg_frac(mv, a, dfn, st[s]["boosts"], st[o]["boosts"], st[s]["status"], chart)
                dm[s].append(f)
                se[s] += mult >= 2; res[s] += 0 < mult <= 0.5; imm[s] += mult == 0
                stab_n[s] += mv["type"] in a["types"]
            else:
                dm[s].append(0.0)
        h1 = sum(last_hp["p1"].values()) + 6 - len(last_hp["p1"])
        h2 = sum(last_hp["p2"].values()) + 6 - len(last_hp["p2"])
        diffs.append(h1 - h2)
    for s in ("p1", "p2"):
        a = np.array(dm[s])
        d[s + "_dmg_total"] = a.sum(); d[s + "_dmg_last10"] = a[-10:].sum(); d[s + "_dmg_max"] = a.max()
        d[s + "_se"] = se[s]; d[s + "_res"] = res[s]; d[s + "_imm"] = imm[s]; d[s + "_stab"] = stab_n[s]
    d["dmg_total_diff"] = d["p1_dmg_total"] - d["p2_dmg_total"]
    d["dmg_last10_diff"] = d["p1_dmg_last10"] - d["p2_dmg_last10"]
    d["se_diff"] = se["p1"] - se["p2"]; d["res_diff"] = res["p2"] - res["p1"]; d["imm_diff"] = imm["p2"] - imm["p1"]

    # team hp difference over time windows
    df_ = np.array(diffs)
    d["hpd_last5"] = df_[-5:].mean(); d["hpd_last10"] = df_[-10:].mean()
    d["hpd_t10"] = df_[9]; d["hpd_t20"] = df_[19]; d["hpd_std"] = df_.std()
    w = np.exp(np.linspace(-2, 0, len(df_))); d["hpd_expw"] = float((df_ * w).sum() / w.sum())

    # hp-weighted strength of what is left (unseen p2 pokemon get the average strength)
    for s in ("p1", "p2"):
        rem, alive = 0.0, 0
        for n, h in last_hp[s].items():
            if h > 0:
                pen = 0.6 if last_status[s].get(n) in ("slp", "frz", "brn", "par") else 1.0
                rem += h * D(n)["bst"] * pen; alive += 1
        unseen = 6 - len(last_hp[s])
        d[s + "_rem_bst"] = rem + unseen * mean["bst"]; d[s + "_alive"] = alive + unseen
    d["rem_bst_diff"] = d["p1_rem_bst"] - d["p2_rem_bst"]; d["alive_diff"] = d["p1_alive"] - d["p2_alive"]
    d["rem_bst_ratio"] = (d["p1_rem_bst"] + 10) / (d["p2_rem_bst"] + 10)

    # last-turn matchup with real stats, boosts, status and the best move each active pokemon has shown
    last = tl[-1]; s1, s2 = last["p1_pokemon_state"], last["p2_pokemon_state"]
    a1, a2 = D(s1["name"]), D(s2["name"])
    best = {}
    for s, sa, so, aa, ao in (("p1", s1, s2, a1, a2), ("p2", s2, s1, a2, a1)):
        fr = [dmg_frac(m, aa, ao, sa["boosts"], so["boosts"], sa["status"], chart)[0] for m in seen_moves[s][sa["name"]].values()]
        best[s] = max(fr) if fr else 0.0
    d["last_best_dmg_p1"] = best["p1"]; d["last_best_dmg_p2"] = best["p2"]
    ttk1 = min(max(s2["hp_pct"], 0.01) / max(best["p1"], 0.02), 20)
    ttk2 = min(max(s1["hp_pct"], 0.01) / max(best["p2"], 0.02), 20)
    d["last_ttk_p1"] = ttk1; d["last_ttk_p2"] = ttk2; d["last_ttk_diff"] = ttk2 - ttk1
    sp1 = spe_eff(a1, s1["boosts"], s1["status"]); sp2 = spe_eff(a2, s2["boosts"], s2["status"])
    d["last_outspeeds"] = float(np.sign(sp1 - sp2)); d["last_spe_ratio"] = sp1 / max(sp2, 1)
    d["last_type_adv"] = (max([tmult(chart, t, a2["types"]) for t in a1["types"]] or [1.0])
                          - max([tmult(chart, t, a1["types"]) for t in a2["types"]] or [1.0]))
    d["last_bst_diff"] = a1["bst"] - a2["bst"]; d["last_hp_diff"] = s1["hp_pct"] - s2["hp_pct"]

    # categorical state descriptions (CatBoost encodes these itself, safely, inside each training fit)
    p1_best = max([n for n in team_names if last_hp["p1"].get(n, 1.0) > 0] or [team_names[0]],
                  key=lambda n: last_hp["p1"].get(n, 1.0) * D(n)["bst"])
    p2_alive_names = [n for n, h in last_hp["p2"].items() if h > 0]
    p2_best = max(p2_alive_names or [s2["name"]], key=lambda n: last_hp["p2"].get(n, 1.0) * D(n)["bst"])
    k1 = sum(h == 0 for h in last_hp["p1"].values()); k2 = sum(h == 0 for h in last_hp["p2"].values())
    m1 = last["p1_move_details"]["name"] if last["p1_move_details"] else "none"
    m2 = last["p2_move_details"]["name"] if last["p2_move_details"] else "none"
    cats["c_lead_pair"] = team_names[0] + "|" + lead2["name"]
    cats["c_last_pair"] = s1["name"] + "|" + s2["name"]
    cats["c_last_status"] = s1["status"] + "|" + s2["status"]
    cats["c_last_move_pair"] = m1 + "|" + m2
    cats["c_best_alive_pair"] = p1_best + "|" + p2_best
    cats["c_ko_state"] = f"{k1}|{k2}"
    cats["c_hp_tier"] = f"{round(s1['hp_pct'] * 4)}|{round(s2['hp_pct'] * 4)}"
    cats["c_last_p1"] = s1["name"]; cats["c_last_p2"] = s2["name"]
    return d, cats

chart = load_chart(TYPE_CHART_PATH)
dex, mean_mon = build_dex(train_path)

def build_new(path):
    N, C = [], []
    with open(path) as f:
        for line in f:
            n, c = new_features(json.loads(line), dex, mean_mon, chart)
            N.append(n); C.append(c)
    return pd.DataFrame(N), pd.DataFrame(C)

new_train, cat_train = build_new(train_path)
new_test, cat_test = build_new(test_path)

# final matrices: base features (cell A) + new numeric + categorical
num_cols = feature_cols + list(new_train.columns)
cat_cols = list(cat_train.columns)
X_all = pd.concat([train_df[feature_cols], new_train, cat_train], axis=1)
X_test_all = pd.concat([test_df[feature_cols], new_test, cat_test], axis=1)
y = train_df["won"].values
print(X_all.shape, X_test_all.shape, "missing:", int(X_all[num_cols].isna().sum().sum()), int(X_test_all[num_cols].isna().sum().sum()))


(10000, 169) (5000, 169) missing: 0 0


In [3]:
#Cell C: honest validation + final model + submission
import os, numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from catboost import CatBoostClassifier

# Every setting below is FIXED in advance. No early stopping on validation data, no label-based feature
# selection, no target encoding, no blend weights or thresholds tuned on validation predictions.
def make_models():
    return {
        "cat5": CatBoostClassifier(iterations=800, learning_rate=0.04, depth=5, l2_leaf_reg=6, random_seed=42,
                                   verbose=0, allow_writing_files=False),
        "cat6": CatBoostClassifier(iterations=1500, learning_rate=0.02, depth=6, l2_leaf_reg=8, random_seed=43,
                                   verbose=0, allow_writing_files=False),
        "logreg": make_pipeline(StandardScaler(), LogisticRegression(C=0.03, max_iter=3000)),
    }

def fit_predict(X_tr, y_tr, X_ap):
    """Fit the 3 models on (X_tr, y_tr) and return the equal-weight average probability for X_ap."""
    probs = []
    for name, m in make_models().items():
        if name == "logreg":
            m.fit(X_tr[num_cols], y_tr); probs.append(m.predict_proba(X_ap[num_cols])[:, 1])
        else:
            m.fit(X_tr, y_tr, cat_features=cat_cols); probs.append(m.predict_proba(X_ap)[:, 1])
    return np.mean(probs, axis=0)

# 1) Cross-validation on the training battles only (the test file is never touched)
skf = StratifiedKFold(5, shuffle=True, random_state=42)
oof = np.zeros(len(y))
for tr, va in skf.split(X_all, y):
    oof[va] = fit_predict(X_all.iloc[tr], y[tr], X_all.iloc[va])
print("5-fold CV accuracy of the blend:", round(((oof > 0.5) == y).mean(), 4))

# 2) Final model on ALL training battles, predict test, write the submission
test_proba = fit_predict(X_all, y, X_test_all)
test_pred = (test_proba > 0.5).astype(int)
print("share predicted p1 win:", round(test_pred.mean(), 3))

pred_by_id = dict(zip(test_df["battle_id"], test_pred))
sample_path = DATA_DIR + "sample_submission.csv"
if os.path.exists(sample_path):
    sub = pd.read_csv(sample_path)
    sub[sub.columns[1]] = sub[sub.columns[0]].map(pred_by_id).astype(int)
else:
    sub = pd.DataFrame({"battle_id": test_df["battle_id"], "player_won": test_pred})
sub.to_csv("submission.csv", index=False)
print(sub.shape); print(sub.head())


5-fold CV accuracy of the blend: 0.6799
share predicted p1 win: 0.514
(5000, 2)
   battle_id  player_won
0          0           1
1          1           0
2          2           0
3          3           0
4          4           1
